# TCAD-Driven Traveling-Wave Mach-Zehnder Modulator

This notebook reproduces the full electro-optic modulator workflow requested in
[gdsfactory/gsim#181](https://github.com/gdsfactory/gsim/issues/181) on
open-source solvers, following the classic CHARGE → MODE → circuit flow:

1. **Charge transport** (`gsim.tcad`, DEVSIM): Poisson + drift-diffusion on the
   waveguide cross-section gives carrier maps $n(x,y)$, $p(x,y)$ and the
   junction capacitance $C(V)$ per bias point.
2. **Validation**: the TCAD $C(V)$ is cross-checked against the retained
   analytic depletion model (`PNJunctionConfig`, Sze).
3. **Carrier→material coupling** (`gsim.common.carriers`): Soref/Nedeljkovic
   plasma dispersion gives $\Delta n(x,y)$, $\Delta\alpha(x,y)$ for optics and
   the Drude $\sigma(x,y)$ for RF.
4. **Carrier-aware modes** (`gsim.femwell`, `gsim.palace`): the optical mode is
   solved with a continuous carrier-perturbed $\varepsilon(x,y)$, and the RF
   mode on the *staircase* — the same carrier maps binned into
   piecewise-constant strips, which is the only form Palace can express.
5. **TW-MZM assembly** (`gsim.common.twmzm`): the RF line parameters and the
   optical bias sweep combine into the traveling-wave transfer function —
   velocity mismatch, impedance, RF loss, EO bandwidth and $V_\pi L$.

All five run as **Stages of one `Study`**. A Stage owns its own cross-section
window, its own route, and its own result; it solves lazily, caches what it
solved, and drops that cache when anything upstream of it changes. So nothing
below moves data between solvers by hand: meshes, physical-group tags and
carrier interpolation all live behind the Stage.

Install everything with the combined extra:

```bash
pip install 'gsim[modulator]'
```

## The device, and a Study over it

A lateral PN junction in a 220 nm silicon rib, split into four doped regions
along the junction axis: `n_pad | n_rib | p_rib | p_pad`, with a metal
electrode landing on each outer pad. `demo_phase_shifter` draws it so this
notebook has something to point at — **in your own workflow the component, the
layer stack and the device description are yours**, and everything below is
unchanged.

`pn_phase_shifter` reads the device description against the drawn
cross-section and configures all five Stages from it: which regions are p and
which are n is all it is told, and the contacts, the region-region interfaces,
the junction and each Stage's window are derived.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from gsim.modulator import demo_phase_shifter, pn_phase_shifter

WAVELENGTH_UM = 1.55
REVERSE_BIASES = [0.0, 0.5, 1.0, 1.5, 2.0]  # V on the cathode = reverse bias
RF_FREQS_HZ = [10e9, 20e9, 40e9, 60e9, 80e9, 100e9]
N_GROUP_OPT = 3.8  # optical group index, for the velocity-mismatch analysis
LENGTH_UM = 3000.0  # traveling-wave electrode length
N_STRIPS = 5  # strips the RF staircase bins the carrier maps into

demo = demo_phase_shifter()  # your own component and stack go here

study = pn_phase_shifter(
    component=demo.component,
    stack=demo.stack,
    device=demo.device,
    biases=REVERSE_BIASES,
    wavelength_um=WAVELENGTH_UM,
    frequencies_hz=RF_FREQS_HZ,
    n_strips=N_STRIPS,
    length_um=LENGTH_UM,
    n_group=N_GROUP_OPT,
    output_dir="./tcad-twmzm",
    verbose=True,
)

layout = study.layout
print("doped regions:", study.device.doped_regions)
print("contacts:     ", [(c.name, c.electrode, c.region) for c in layout.contacts])
print("interfaces:   ", [i.name for i in layout.interfaces])
print(f"junction:      {layout.junction.regions} at y = {layout.junction_position} um")

### One component, three cross-section windows

Each Stage clips the cross-section to what its own physics needs, and derives
that window from the device rather than being told it: charge transport spans
the doped slab between the contacts, the optical mode a few-µm box around the
rib, and the RF solve keeps the full extent of the staircase it builds.

In [ ]:
optical_window = layout.window_around_junction(margin_um=study.optical.mode_margin_um)
optical_window_z = layout.window_z_around_guide(
    above_um=study.optical.z_above_um, below_um=study.optical.z_below_um
)
print(f"charge  window: y in {layout.window} um (doped slab between contacts)")
print(f"optical window: y in {optical_window} um, z in {optical_window_z} um")
print(f"rf      window: {study.rf.window} (None = the full cross-section)")
print(f"rf strips tile: y in {study.rf.strip_span} um (the doped slab, pads included)")

## Charge transport: Poisson + drift-diffusion (DEVSIM)

Running the charge Stage meshes its window and sweeps the bias. Positive
cathode (n-side) bias reverse-biases the junction. Each bias point carries the
carrier maps on the mesh nodes, the terminal currents, and the small-signal
capacitance from a quasi-static AC solve.

In [ ]:
sweep = study.charge.run()

for point in sweep.points:
    print(
        f"V = {point.bias_v:4.1f} V:  C = {point.capacitance_f_per_m * 1e12:6.1f} pF/m,"
        f"  I_cathode = {point.currents_a_per_cm['cathode']:+.2e} A/cm"
    )

## TCAD $C(V)$ versus the analytic depletion model

The retained `PNJunctionConfig` (Sze) path is the quick analytic estimate and
doubles as a validation reference: on this abrupt symmetric junction the
numeric small-signal $C(V)$ must track $\varepsilon_s / W(V)$.

In [ ]:
from gsim import tcad
from gsim.common.stack.junction import PNJunctionConfig

junction = PNJunctionConfig(
    na_cm3=study.device.p_doping_cm3, nd_cm3=study.device.n_doping_cm3
)
comparison = tcad.compare_capacitance(junction, sweep, height_um=demo.rib_height_um)
print(f"max relative deviation: {comparison.max_relative_deviation:.1%}")

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(
    comparison.v_reverse, comparison.c_tcad_f_per_cm * 1e14, "o-", label="TCAD (DEVSIM)"
)
ax.plot(
    comparison.v_reverse,
    comparison.c_analytic_f_per_cm * 1e14,
    "s--",
    label="Analytic (Sze)",
)
ax.set_xlabel("Reverse bias (V)")
ax.set_ylabel("C (fF per um of length)")
ax.set_title("Junction capacitance: TCAD vs depletion approximation")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

### Carrier maps

The depletion region widens with reverse bias — the physics the single-strip
depletion model cannot resolve spatially.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3), sharey=True)
for ax, point in zip(axes, (sweep.points[0], sweep.points[-1]), strict=False):
    carriers = point.carriers
    tri = ax.tricontourf(
        carriers.x_um,
        carriers.y_um,
        np.log10(np.maximum(carriers.electrons_cm3, 1.0)),
        levels=30,
    )
    ax.set_title(f"log10 n(x, y) at V = {point.bias_v} V")
    ax.set_xlabel("y (um)")
fig.colorbar(tri, ax=axes, label="log10 n (cm^-3)")
axes[0].set_ylabel("z (um)")
plt.show()

## Carrier → material coupling

The carriers Stage evaluates the coupling at every node of every bias point:
the plasma-dispersion index shift $\Delta n$ and free-carrier absorption
$\Delta\alpha$ for optics, and the Drude $\sigma = q(\mu_n n + \mu_p p)$ for
RF. The coefficients are a Stage setting — `study.carriers(dispersion=...)` —
so foundry-calibrated values substitute for the published silicon fits.

In [ ]:
response = study.carriers.run()
print("coefficients fitted at:", study.carriers.dispersion.wavelength_um, "um")

fig, axes = plt.subplots(1, 3, figsize=(13, 3.2))
for point in (response.points[0], response.points[-1]):
    carriers = point.carriers
    band = np.abs(carriers.y_um - demo.rib_height_um / 2) < 0.03
    order = np.argsort(carriers.x_um[band])
    y = carriers.x_um[band][order]
    label = f"V = {point.bias_v} V"
    axes[0].plot(y, point.index_shift[band][order], label=label)
    axes[1].plot(y, point.absorption_cm[band][order], label=label)
    axes[2].plot(y, point.conductivity_s_per_m[band][order], label=label)
axes[0].set_ylabel("$\\Delta n$")
axes[1].set_ylabel("$\\Delta\\alpha$ (1/cm)")
axes[2].set_ylabel("$\\sigma$ (S/m)")
for ax in axes:
    ax.set_xlabel("y (um)")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
fig.suptitle("Carrier-derived material response across the junction (mid-rib cut)")
plt.tight_layout()
plt.show()

## Carrier-aware optical modes

The optical Stage meshes its own rib-box window, carries the carrier maps of
every bias point onto that mesh, and solves with a continuous
$\varepsilon(x,y)$: every element of a doped region takes
$(n_{Si} + \Delta n)^2$ with the local absorption as a negative imaginary part.
That is the femwell route, and it is the reference for a graded profile —
Palace takes piecewise-constant materials per region and nothing else, so
routing this Stage to it (`study.optical(route="palace", n_strips=...)`) puts
it on a staircase instead.

In [ ]:
optical = study.optical.run()

for mode in optical.points:
    print(
        f"V = {mode.bias_v:4.1f} V:  n_eff = {mode.n_eff:.6f},  "
        f"dn_eff = {mode.index_shift:+.2e},  loss = {mode.loss_db_cm:5.2f} dB/cm"
    )

### Phase-shift efficiency $V_\pi L$ and bias-dependent optical loss

The mode/carrier overlap is fully resolved, so the efficiency reflects the true
depletion-edge movement rather than a uniform-strip estimate.

In [ ]:
from gsim.common.twmzm import vpi_length_vcm

vpi_l = vpi_length_vcm(
    optical.voltages, optical.index_shift, wavelength_um=optical.wavelength_um
)

fig, axes = plt.subplots(1, 3, figsize=(13, 3.2))
axes[0].plot(optical.voltages, optical.index_shift * 1e5, "o-")
axes[0].set_ylabel("$\\Delta n_{eff} \\times 10^{-5}$")
axes[1].plot(optical.voltages, vpi_l, "o-")
axes[1].set_ylabel("$V_\\pi L$ (V cm)")
axes[2].plot(optical.voltages, optical.loss_db_cm, "o-")
axes[2].set_ylabel("optical loss (dB/cm)")
for ax in axes:
    ax.set_xlabel("Reverse bias (V)")
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()
print(f"V_pi L at {optical.voltages[-2]:.1f} V: {vpi_l[-2]:.2f} V cm")

## The staircase the RF Stage solves

Both EM backends take piecewise-constant materials per region, so the RF Stage
reduces the chosen bias point's carrier map to $N$ strips along the junction
axis — each carrying the Drude conductivity of its average carrier
concentrations — and draws them as a cross-section of its own, flanked by the
electrode's two conductors. The strips span the whole doped slab, so the pads
and their series resistance are part of the line. $N = 1$ recovers the
uniform-strip depletion model; increasing $N$ converges to the continuous
profile.

In [ ]:
staircase = study.rf.staircase()

edges = np.asarray(staircase.strips["edges_um"], dtype=float)
sigma = np.asarray(staircase.strips["sigma_s_per_m"], dtype=float)
print("strip regions:  ", staircase.strip_names)
print("strip edges (um):", np.round(edges, 3))
print("strip sigma (S/m):", np.round(sigma, 1))
print("electrodes (um):", staircase.electrode_spans)
print("signal conductor:", study.rf.signal_electrode())

fig, ax = plt.subplots(figsize=(6, 3))
ax.stairs(sigma, edges, fill=True, alpha=0.6)
ax.set_xlabel("y (um)")
ax.set_ylabel("$\\sigma$ (S/m)")
ax.set_title(f"Staircase conductivity at V = {study.rf.bias_point().bias_v} V")
ax.grid(alpha=0.3)
plt.show()

## RF line parameters with carrier-derived $\sigma(x,y)$

The RF Stage meshes that staircase once and solves the line mode at every
frequency, tracking it across the sweep so the shift-invert search cannot
settle on a different branch as the materials move. The complex $n_{eff}$ gives
$\gamma = \alpha + i\beta$, and $Z_0$ comes from the Marks-Williams
power-current integral $Z_0 = 2P/|I|^2$ with $I$ integrated over the signal
conductor — no analytic line model anywhere in this chain.

The same staircase solves on Palace with `study.rf(route="palace")`, which
reports the effective index and leaves the impedance NaN: Palace's
boundary-mode results carry no mode fields to integrate.

In [ ]:
line_params = study.rf.run()

for freq, n_rf, alpha, z0 in zip(
    line_params.freq_hz,
    line_params.n_rf,
    line_params.alpha_rf_np_m,
    line_params.z0_ohm,
    strict=True,
):
    print(
        f"f = {freq / 1e9:5.0f} GHz:  n_RF = {n_rf:5.2f},  "
        f"alpha_RF = {alpha * 8.686 / 100:5.1f} dB/cm,  Z0 = {z0.real:5.1f} ohm"
    )
print(f"staircased at V = {study.rf.solved_bias_v} V")

## Velocity-mismatch analysis

The explicit "sample of velocity mismatch" from #181: the solved RF index
against the optical group index, and the walk-off-limited bandwidth
$f_{3dB} = 1.39\,c/(\pi L |n_{RF} - n_g|)$ as a function of electrode length.
Velocity mismatch caps the bandwidth on its own, before any RF loss or
reflection is counted.

In [ ]:
from gsim.common.twmzm import walkoff_bandwidth

n_rf_mean = float(np.mean(line_params.n_rf))
mismatch = abs(n_rf_mean - N_GROUP_OPT)
lengths_mm = np.linspace(1, 10, 40)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].plot(line_params.freq_hz / 1e9, line_params.n_rf, "o-", label="$n_{RF}$ solved")
axes[0].axhline(N_GROUP_OPT, color="k", ls="--", label="$n_g$ optical")
axes[0].set_xlabel("frequency (GHz)")
axes[0].set_ylabel("index")
axes[0].set_title("Velocity matching vs frequency")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(
    lengths_mm,
    [
        walkoff_bandwidth(length_m=length * 1e-3, n_rf=n_rf_mean, n_opt=N_GROUP_OPT)
        / 1e9
        for length in lengths_mm
    ],
)
axes[1].axvline(study.line.length_um * 1e-3, color="r", ls="--", label="this device")
axes[1].set_xlabel("electrode length (mm)")
axes[1].set_ylabel("walk-off $f_{3dB}$ (GHz)")
axes[1].set_title(f"Walk-off limit ($|n_{{RF}} - n_g|$ = {mismatch:.2f})")
axes[1].legend()
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Full device report: EO bandwidth and figures of merit

The line Stage adds the two things that are not cross-section physics — how
long the electrode is and what it is driven from and terminated into — and
combines the optical and RF results into the standard single-drive
traveling-wave response, including RF loss and source/load reflections. Asking
the Study for its report runs whatever has not run yet; here both EM Stages
already have.

In [ ]:
study.line(
    z_load_ohm=float(np.mean(line_params.z0_ohm.real)),  # terminate in the line
    z_gen_ohm=50.0,
    # A denser grid than the RF stage solved, so the 3 dB point is
    # readable; kept inside the solved range, which is not extrapolated.
    response_frequencies_hz=list(np.linspace(min(RF_FREQS_HZ), max(RF_FREQS_HZ), 200)),
)
report = study.report()

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(report.freq_hz / 1e9, 20 * np.log10(np.abs(report.response)))
ax.axhline(20 * np.log10(1 / np.sqrt(2)), color="k", ls=":", label="-3 dB (EO)")
if report.bandwidth_3db_hz:
    ax.axvline(report.bandwidth_3db_hz / 1e9, color="r", ls="--")
ax.set_xlabel("frequency (GHz)")
ax.set_ylabel("EO response (dB)")
ax.set_title(f"TW-MZM response, L = {report.length_m * 1e3:.0f} mm, matched load")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

print("=== TW-MZM figures of merit ===")
if report.bandwidth_3db_hz:
    print(f"EO 3 dB bandwidth:        {report.bandwidth_3db_hz / 1e9:.1f} GHz")
if report.walkoff_bandwidth_hz:
    print(f"walk-off limit:           {report.walkoff_bandwidth_hz / 1e9:.1f} GHz")
print(f"velocity mismatch:        n_RF - n_g = {report.velocity_mismatch[0]:+.2f}")
print(f"characteristic impedance: {report.z0_ohm[0].real:.1f} ohm")
print(
    f"V_pi L at {report.voltages_v[-2]:.1f} V:        {report.vpi_l_vcm[-2]:.2f} V cm"
)
rlgc = report.rlgc
print(
    f"RLGC at 10 GHz: R = {np.interp(10e9, report.freq_hz, rlgc['R']):.0f} ohm/m, "
    f"L = {np.interp(10e9, report.freq_hz, rlgc['L']) * 1e9:.0f} nH/m, "
    f"C = {np.interp(10e9, report.freq_hz, rlgc['C']) * 1e12:.0f} pF/m"
)

## Summary

From one gdsfactory component, one layer stack and a device description naming
which regions are p and which are n:

- **DEVSIM** solved the junction charge transport per bias point — carrier maps
  and $C(V)$, validated against the analytic Sze model;
- the **carriers Stage** turned those maps into the index shift, the
  free-carrier absorption and the Drude conductivity;
- **femwell** solved the optical mode with the continuous carrier-perturbed
  index — $\Delta n_{eff}(V)$, optical loss and $V_\pi L$ — and the RF line
  mode of the electrode-loaded staircase, $\gamma$ from the solver and $Z_0$
  from the Marks-Williams power-current integral;
- the **line Stage** produced the velocity-mismatch analysis, the RLGC line
  parameters and the electro-optic bandwidth.

Every Stage derived its own window, its own mesh and its own materials from the
same device description. Nothing above read a mesh file, mapped a physical-group
tag, interpolated a carrier map, or assembled a second component — those are the
Stage's job, and swapping a route (`study.rf(route="palace")`) or a coefficient
set (`study.carriers(dispersion=...)`) does not change a line of this notebook.

This is the open-source counterpart of the commercial CHARGE → MODE → circuit
modulator workflow requested in gdsfactory/gsim#181.